# Décodage — imports

Charge le modèle et les fonctions de décodage.

In [1]:
import sys
from pathlib import Path

import torch

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.model import load_model
from src.llm.decoding import (
    apply_temperature,
    apply_top_k,
    apply_top_p,
    sample_next_token,
)

# Chargement du modèle

Récupère le tokenizer, le modèle et le device.

In [2]:
llm = load_model()

tokenizer = llm.tokenizer
model = llm.model
device = llm.device

Chargement du modèle : Qwen/Qwen2.5-0.5B-Instruct
Device utilisé : cpu


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

# Logits du prochain token

Calcule les scores produits après le prompt.

In [3]:
text = "Paris est la capitale de"

inputs = tokenizer(
    text,
    return_tensors="pt",
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[0, -1, :]

logits.shape

torch.Size([151936])

# Température

Observe son effet sur la distribution des probabilités.

In [4]:
temperatures = [0.2, 0.7, 1.0, 1.5, 2.0]

for temperature in temperatures:

    scaled_logits = apply_temperature(
        logits,
        temperature,
    )

    probabilities = torch.softmax(
        scaled_logits,
        dim=-1,
    )

    top_probs, top_ids = torch.topk(
        probabilities,
        k=5,
    )

    print(f"\nTemperature = {temperature}")

    for prob, token_id in zip(
        top_probs.tolist(),
        top_ids.tolist(),
    ):
        print(
            repr(tokenizer.decode([token_id])),
            round(prob, 4),
        )


Temperature = 0.2
' la' 0.9531
' l' 0.0474
' France' 0.0
' ce' 0.0
' qui' 0.0

Temperature = 0.7
' la' 0.6484
' l' 0.3066
' France' 0.0284
' ce' 0.0044
' qui' 0.0021

Temperature = 1.0
' la' 0.4844
' l' 0.2754
' France' 0.051
' ce' 0.0146
' qui' 0.0083

Temperature = 1.5
' la' 0.1025
' l' 0.0703
' France' 0.0229
' ce' 0.0098
' qui' 0.0068

Temperature = 2.0
' la' 0.0168
' l' 0.0127
' France' 0.0055
' ce' 0.0029
' qui' 0.0022


# Top-k

Ne conserve que les `k` tokens les mieux classés.

In [5]:
filtered_logits = apply_top_k(
    logits,
    k=5,
)

probabilities = torch.softmax(
    filtered_logits,
    dim=-1,
)

non_zero = torch.where(
    probabilities > 0
)[0]

print("Nombre de tokens possibles :", len(non_zero))

for token_id in non_zero:
    print(
        repr(tokenizer.decode([token_id.item()])),
        probabilities[token_id].item(),
    )

Nombre de tokens possibles : 5
' l' 0.330078125
' la' 0.58203125
' ce' 0.017578125
' qui' 0.010009765625
' France' 0.061279296875


# Top-p

Conserve les tokens jusqu’au seuil de probabilité cumulée.

In [6]:
filtered_logits = apply_top_p(
    logits,
    p=0.9,
)

probabilities = torch.softmax(
    filtered_logits,
    dim=-1,
)

remaining = torch.where(
    probabilities > 0
)[0]

print(
    "Nombre de tokens conservés :",
    len(remaining),
)

Nombre de tokens conservés : 52


# Sampling

Échantillonne plusieurs tokens selon la distribution filtrée.

In [7]:
for _ in range(10):

    token_id = sample_next_token(
        logits,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
    )

    print(
        repr(
            tokenizer.decode(
                token_id.tolist()
            )
        )
    )

' la'
' la'
' l'
' la'
' la'
' l'
' l'
' l'
' l'
' la'


# Génération complète

Compare plusieurs températures sur une réponse complète.

In [8]:
from src.llm.decoding import generate


prompt = "Explique simplement ce qu'est un Transformer."
print("température faible")
print(
    generate(
        model,
        tokenizer,
        prompt,
        device,
        temperature=0.2,
        top_k=50,
        top_p=0.9,
    )
)

print("température moyenne")
print(
    generate(
        model,
        tokenizer,
        prompt,
        device,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
    )
)

print("température élevée")
print(
    generate(
        model,
        tokenizer,
        prompt,
        device,
        temperature=1.5,
        top_k=50,
        top_p=0.9,
    )
)

température faible
Explique simplement ce qu'est un Transformer. Un Transformer est un modèle de neurones qui est capable de transformer les données en une forme plus simple. C'est un type de réseau de neurones qui utilise des méthodes de réduction de l'entropie pour transformer les données en une
température moyenne
Explique simplement ce qu'est un Transformer. Un transformer est une algorithme de traitement de l'information (AI) qui utilise le modèle du transformer pour transformer les données de manière linéaire à une nouvelle forme linéaire. C'est une méthode de traitement de données qui utilise l'
température élevée
Explique simplement ce qu'est un Transformer. Transformer, ou modèle transpositionne de forme, ou simply d'application, se base sur la structure linguistique. Cela se traduit par la réunion d'éléments complexes. Ici se trouve en deux parties.
1° C
